# MBGCN — Multi-Behavior Graph Convolutional Network

**Nguồn model:** [tsinghua-fib-lab/MBGCN](https://github.com/tsinghua-fib-lab/MBGCN) (Jin et al., SIGIR 2020).

**Dataset:** `nguyenmaiductrong/rees46-bpatmp-temporal` (HuggingFace) — behaviors: `view, cart, purchase`; target = `purchase`.

### Hai thành phần cốt lõi của MBGCN (đều được giữ nguyên)
1. **User-based CF** — lan truyền trên đồ thị user–item đa hành vi, gộp neighbor theo trọng số
   *behavior-aware* `mgnn_weight × user_behaviour_degree`. → `score1`
2. **Item-based CF** — lan truyền trên **đồ thị item–item đồng xuất hiện** (`item_graph`) của từng hành vi,
   chiếu qua `item_behaviour_W`. → `score2`
3. Điểm cuối: `score = score1 + λ · score2`. Loss: BPR + L2.

### Khác biệt so với repo gốc (đã ghi chú rõ để fair)
| Hạng mục | Repo gốc (Tmall) | Notebook này |
|---|---|---|
| Embedding khởi tạo | pretrain MF rồi load | xavier ngẫu nhiên (không có pretrain) |
| Đồ thị item–item | precompute đầy đủ (catalog nhỏ) | **top-K neighbor** mỗi item (catalog 29.9k item → O(item²) không khả thi nếu đầy đủ) |
| Negative sampling | pre-sample ra file mỗi epoch | on-the-fly trong DataLoader |
| Đánh giá | Recall/NDCG/MRR riêng của repo | **HR@k / NDCG@k khớp `TemporalSplitEvaluator` của dự án** (so sánh được với baseline khác) |



In [1]:
# CELL 1: CÀI ĐẶT THƯ VIỆN
!pip install -q huggingface_hub

In [2]:
# ============================================================
# CELL 2: IMPORT
# ============================================================
import os, json, time, random, pickle, gc
from collections import defaultdict

import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from huggingface_hub import snapshot_download

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if device.type != 'cuda':
    print("Không có GPU — MBGCN lan truyền toàn đồ thị mỗi step, chạy CPU sẽ rất chậm.")

Device: cuda


In [3]:
# ============================================================
# CELL 3: TẢI DATA TỪ HUGGING FACE
# ============================================================
RAW_DIR = "./rees46_raw_data"
os.makedirs(RAW_DIR, exist_ok=True)

REPO_ID = "nguyenmaiductrong/rees46-bpatmp-temporal"
need = ["node_counts.json",
        "val_ground_truth.pkl", "test_ground_truth.pkl",
        "train_mask_purchase_only.pkl"]
have_all = all(os.path.exists(os.path.join(RAW_DIR, f)) for f in need) and \
           any(f.endswith('.npy') for f in os.listdir(RAW_DIR))

if not have_all:
    print("Đang tải data từ HuggingFace...")
    snapshot_download(
        repo_id=REPO_ID,
        repo_type="dataset",
        allow_patterns=["*.npy", "*.pkl", "node_counts.json"],
        local_dir=RAW_DIR,
    )
    print("Tải hoàn tất!")
else:
    print(f"Data đã có trong {RAW_DIR}")

print("\nFile .npy / .pkl:")
for f in sorted(os.listdir(RAW_DIR)):
    if f.endswith(('.npy', '.pkl')):
        print(f"  {f}")

Đang tải data từ HuggingFace...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 22 files:   0%|          | 0/22 [00:00<?, ?it/s]

Tải hoàn tất!

File .npy / .pkl:
  candidate_item_idx.npy
  cart_train_dst.npy
  cart_train_src.npy
  cart_train_ts.npy
  purchase_train_dst.npy
  purchase_train_src.npy
  purchase_train_ts.npy
  test_ground_truth.pkl
  test_product_idx.npy
  test_timestamp.npy
  test_user_idx.npy
  train_mask.pkl
  train_mask_purchase_only.pkl
  train_mask_seen_all.pkl
  val_ground_truth.pkl
  val_product_idx.npy
  val_timestamp.npy
  val_user_idx.npy
  view_train_dst.npy
  view_train_src.npy
  view_train_ts.npy


In [4]:
# ============================================================
# CELL 4: LOAD BEHAVIORS  (target = purchase, đặt ĐẦU TIÊN theo quy ước MBGCN)
# ============================================================
BEHAVIORS = ['purchase', 'cart', 'view']   # relation list — target trước
TARGET    = 'purchase'

# Số user/item lấy từ node_counts.json (authoritative — khớp index trong eval)
with open(f"{RAW_DIR}/node_counts.json") as f:
    NC = json.load(f)
# node_counts.json là flat: {"user":..,"product":..,"category":..,"brand":..}
# (giống cách scripts/run_training.py đọc node_counts["user"])
NUM_USERS = int(NC["user"])
NUM_ITEMS = int(NC["product"])

def load_behavior_train(bhv):
    u = np.load(f"{RAW_DIR}/{bhv}_train_src.npy")
    i = np.load(f"{RAW_DIR}/{bhv}_train_dst.npy")
    t = np.load(f"{RAW_DIR}/{bhv}_train_ts.npy")
    return u, i, t

behavior_data = {}
for b in BEHAVIORS:
    u, i, t = load_behavior_train(b)
    behavior_data[b] = {'user': u, 'item': i, 'ts': t}
    print(f"[{b:8s}] {len(u):>12,} interactions | u_max={u.max():,} | i_max={i.max():,}")

print(f"\nTổng (từ node_counts): {NUM_USERS:,} users | {NUM_ITEMS:,} items")

[purchase]    2,436,760 interactions | u_max=203,062 | i_max=29,891
[cart    ]    3,868,050 interactions | u_max=203,061 | i_max=29,891
[view    ]   25,956,733 interactions | u_max=203,061 | i_max=29,891

Tổng (từ node_counts): 203,063 users | 29,892 items


In [5]:
# ============================================================
# CELL 5: XÂY CÁC CẤU TRÚC ĐỒ THỊ CHO MBGCN
#   - relation_dict[b]       : sparse (user × item) cho từng hành vi
#   - train_matrix           : sparse (user × item) của TARGET (purchase)
#   - user_behaviour_degree  : (user × n_behaviors) — số tương tác/hành vi
#   - item_graph[b]          : sparse (item × item) đồng xuất hiện, top-K neighbor
#   - item_graph_degree[b]   : (item × 1) tổng trọng số hàng
# ============================================================
ITEM_GRAPH_TOPK = 10   # số item-neighbor giữ lại mỗi item (giới hạn O(item²))

def build_ui_sparse(users, items, n_u, n_i):
    R = sp.csr_matrix((np.ones(len(users), dtype=np.float32), (users, items)),
                      shape=(n_u, n_i))
    R.sum_duplicates()
    R.data[:] = 1.0            # binarize
    return R

def scipy_to_torch_sparse(mat):
    coo = mat.tocoo()
    idx = torch.LongTensor(np.vstack([coo.row, coo.col]))
    val = torch.FloatTensor(coo.data)
    return torch.sparse_coo_tensor(idx, val, torch.Size(coo.shape)).coalesce()

def build_item_graph_topk(R, topk, block=512):
    """S = R^T R (đồng xuất hiện item-item), bỏ đường chéo, giữ top-K mỗi hàng."""
    n_items = R.shape[1]
    Rt = R.T.tocsr()                       # item × user
    rows, cols, vals = [], [], []
    for start in range(0, n_items, block):
        end = min(start + block, n_items)
        sub = (Rt[start:end] @ R).toarray()        # (b × n_items) đếm đồng xuất hiện
        for r in range(end - start):
            item = start + r
            row = sub[r]
            row[item] = 0.0                        # bỏ self-loop
            nz = np.nonzero(row)[0]
            if nz.size == 0:
                continue
            if nz.size > topk:
                nz = nz[np.argpartition(row[nz], -topk)[-topk:]]
            rows.extend([item] * nz.size)
            cols.extend(nz.tolist())
            vals.extend(row[nz].tolist())
    idx = torch.LongTensor(np.vstack([rows, cols])) if rows else torch.zeros((2, 0), dtype=torch.long)
    val = torch.FloatTensor(vals)
    return torch.sparse_coo_tensor(idx, val, (n_items, n_items)).coalesce()

relation_dict, R_dict = {}, {}
item_graph, item_graph_degree = {}, {}
ubd = np.zeros((NUM_USERS, len(BEHAVIORS)), dtype=np.float32)

print("Đang xây đồ thị (relation + item-item)...")
for bi, b in enumerate(BEHAVIORS):
    t0 = time.time()
    R = build_ui_sparse(behavior_data[b]['user'], behavior_data[b]['item'], NUM_USERS, NUM_ITEMS)
    R_dict[b] = R
    relation_dict[b] = scipy_to_torch_sparse(R)
    ubd[:, bi] = np.asarray(R.sum(axis=1)).ravel()
    S = build_item_graph_topk(R, ITEM_GRAPH_TOPK)
    item_graph[b] = S
    item_graph_degree[b] = torch.sparse.sum(S, dim=1).to_dense().unsqueeze(-1)
    print(f"  [{b:8s}] R nnz={R.nnz:>12,} | item_graph nnz={S._nnz():>9,} | {time.time()-t0:5.1f}s")

user_behaviour_degree = torch.FloatTensor(ubd)          # (n_users, n_behaviors)
train_matrix = relation_dict[TARGET]                    # target behavior
print("\nHoàn tất. user_behaviour_degree:", tuple(user_behaviour_degree.shape))

Đang xây đồ thị (relation + item-item)...


/tmp/ipykernel_1952/3289525578.py:22: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:760.)
  return torch.sparse_coo_tensor(idx, val, torch.Size(coo.shape)).coalesce()


  [purchase] R nnz=   1,308,730 | item_graph nnz=  294,807 |   8.3s
  [cart    ] R nnz=   1,578,188 | item_graph nnz=  291,003 |   8.5s
  [view    ] R nnz=  10,310,160 | item_graph nnz=  298,914 |  30.1s

Hoàn tất. user_behaviour_degree: (203063, 3)


In [6]:
# ============================================================
# CELL 6: MBGCN MODEL  — bám sát model.py của tsinghua-fib-lab/MBGCN
# ============================================================
class MBGCN(nn.Module):
    """MBGCN chuẩn: shared embeddings + user-based CF (score1) + item-based CF (score2).

    forward(user, item):  user (B,1), item (B, 1+num_neg) -> scores (B, 1+num_neg), L2
    Dùng cho training (BPR). Đánh giá dùng prepare_eval() + score_users().
    """
    def __init__(self, num_users, num_items, behaviors,
                 relation_dict, item_graph, item_graph_degree,
                 user_behaviour_degree, train_matrix,
                 embed_size=64, lamb=1.0, L2_norm=1e-4, mgnn_weight=None,
                 node_dropout=0.2, message_dropout=0.2, device='cpu'):
        super().__init__()
        self.device = device
        self.num_users, self.num_items = num_users, num_items
        self.behaviors = list(behaviors)
        self.n_behaviors = len(self.behaviors)
        self.embed_size = embed_size
        self.lamb = lamb
        self.L2_norm = L2_norm

        # ── dữ liệu đồ thị (hằng số, đẩy lên device) ──
        self.relation_dict = {k: v.to(device) for k, v in relation_dict.items()}
        self.item_graph = {k: v.to(device) for k, v in item_graph.items()}
        self.item_graph_degree = {k: v.to(device) for k, v in item_graph_degree.items()}
        self.user_behaviour_degree = user_behaviour_degree.to(device)
        self.train_matrix = train_matrix.to(device)
        self.train_matrix_t = train_matrix.t().coalesce().to(device)

        # ── shared embeddings (1 bảng dùng chung — đúng model_base gốc) ──
        self.user_embedding = nn.Parameter(torch.empty(num_users, embed_size))
        self.item_embedding = nn.Parameter(torch.empty(num_items, embed_size))
        nn.init.xavier_normal_(self.user_embedding)
        nn.init.xavier_normal_(self.item_embedding)

        # ── tham số behavior-aware ──
        if mgnn_weight is None:
            mgnn_weight = [1.0] * self.n_behaviors
        self.mgnn_weight = nn.Parameter(torch.FloatTensor(mgnn_weight))
        self.item_behaviour_W = nn.ParameterList(
            [nn.Parameter(torch.empty(embed_size * 2, embed_size * 2)) for _ in self.behaviors])
        self.item_propagate_W = nn.ParameterList(
            [nn.Parameter(torch.empty(embed_size, embed_size)) for _ in self.behaviors])
        self.W = nn.Parameter(torch.empty(embed_size, embed_size))
        for p in self.item_behaviour_W: nn.init.xavier_normal_(p)
        for p in self.item_propagate_W: nn.init.xavier_normal_(p)
        nn.init.xavier_normal_(self.W)

        self.message_drop = nn.Dropout(p=message_dropout)
        self.train_node_drop = nn.Dropout(p=node_dropout)
        self.node_drop = nn.ModuleList([nn.Dropout(p=node_dropout) for _ in self.behaviors])

    # L2 trên embedding của target được chọn trong batch
    def regularize(self, u_emb, i_emb):
        return self.L2_norm * ((u_emb ** 2).sum() + (i_emb ** 2).sum())

    def _user_behaviour_weight(self):
        weight = self.mgnn_weight.unsqueeze(-1)                      # (B,1)
        total = torch.mm(self.user_behaviour_degree, weight)        # (n_users,1)
        return self.user_behaviour_degree * self.mgnn_weight.unsqueeze(0) / (total + 1e-8)

    @staticmethod
    def _drop_sparse(sp_tensor, drop):
        idx, val = sp_tensor._indices(), sp_tensor._values()
        val = drop(val)
        return torch.sparse_coo_tensor(idx, val, sp_tensor.shape).coalesce()

    def forward(self, user, item):
        # node dropout trên train matrix (target)
        train_matrix = self._drop_sparse(self.train_matrix, self.train_node_drop)
        ubw = self._user_behaviour_weight()

        score2 = None
        user_feature = None
        for i, key in enumerate(self.behaviors):
            rel = self._drop_sparse(self.relation_dict[key], self.node_drop[i])

            item_prop = torch.mm(
                torch.sparse.mm(self.item_graph[key], self.item_embedding)
                / (self.item_graph_degree[key] + 1e-8),
                self.item_propagate_W[i])                            # (n_items, d)
            item_prop = torch.cat((self.item_embedding, item_prop), dim=1)   # (n_items, 2d)
            tmp_item_embedding = item_prop[item]                     # (B, 1+neg, 2d)

            deg_i = self.user_behaviour_degree[:, i].unsqueeze(-1) + 1e-8
            user_neighbour = torch.sparse.mm(rel, self.item_embedding) / deg_i          # (n_users, d)
            user_item_neighbour_p = torch.sparse.mm(rel, item_prop) / deg_i             # (n_users, 2d)

            proj = torch.mm(user_item_neighbour_p, self.item_behaviour_W[i])            # (n_users, 2d)
            tproj = proj[user].expand(-1, item.shape[1], -1)                            # (B, 1+neg, 2d)
            s2 = torch.sum(tproj * tmp_item_embedding, dim=2)                           # (B, 1+neg)

            uf = ubw[:, i].unsqueeze(-1) * user_neighbour
            if i == 0:
                user_feature, score2 = uf, s2
            else:
                user_feature = user_feature + uf
                score2 = score2 + s2
        score2 = score2 / self.n_behaviors

        item_feature = torch.sparse.mm(train_matrix.t().coalesce(), self.user_embedding)  # (n_items, d)
        user_feature = torch.mm(user_feature, self.W)
        item_feature = torch.mm(item_feature, self.W)
        user_feature = torch.cat((self.user_embedding, user_feature), dim=1)              # (n_users, 2d)
        item_feature = torch.cat((self.item_embedding, item_feature), dim=1)              # (n_items, 2d)
        user_feature = self.message_drop(user_feature)
        item_feature = self.message_drop(item_feature)

        tmp_user_feature = user_feature[user].expand(-1, item.shape[1], -1)              # (B, 1+neg, 2d)
        tmp_item_feature = item_feature[item]                                            # (B, 1+neg, 2d)
        score1 = torch.sum(tmp_user_feature * tmp_item_feature, dim=2)                   # (B, 1+neg)

        scores = score1 + self.lamb * score2
        L2 = self.regularize(tmp_user_feature, tmp_item_feature)
        return scores, L2

    # ── đánh giá: precompute toàn bộ feature 1 lần, rồi score theo batch user ──
    @torch.no_grad()
    def prepare_eval(self):
        self.eval()
        ubw = self._user_behaviour_weight()
        self._eval_item_prop, self._eval_proj = [], []
        user_feature = None
        for i, key in enumerate(self.behaviors):
            item_prop = torch.mm(
                torch.sparse.mm(self.item_graph[key], self.item_embedding)
                / (self.item_graph_degree[key] + 1e-8),
                self.item_propagate_W[i])
            item_prop = torch.cat((self.item_embedding, item_prop), dim=1)               # (n_items, 2d)

            deg_i = self.user_behaviour_degree[:, i].unsqueeze(-1) + 1e-8
            un = torch.sparse.mm(self.relation_dict[key], self.item_embedding) / deg_i
            uinp = torch.sparse.mm(self.relation_dict[key], item_prop) / deg_i
            proj = torch.mm(uinp, self.item_behaviour_W[i])                              # (n_users, 2d)

            uf = ubw[:, i].unsqueeze(-1) * un
            user_feature = uf if i == 0 else user_feature + uf
            self._eval_item_prop.append(item_prop)
            self._eval_proj.append(proj)

        item_feature = torch.sparse.mm(self.train_matrix_t, self.user_embedding)
        user_feature = torch.mm(user_feature, self.W)
        item_feature = torch.mm(item_feature, self.W)
        self._eval_user_feat = torch.cat((self.user_embedding, user_feature), dim=1)     # (n_users, 2d)
        self._eval_item_feat = torch.cat((self.item_embedding, item_feature), dim=1)     # (n_items, 2d)

    @torch.no_grad()
    def score_users(self, users):
        """users: LongTensor (B,) -> scores (B, n_items)  (score1 + λ·score2)"""
        uf = self._eval_user_feat[users]                              # (B, 2d)
        score1 = torch.mm(uf, self._eval_item_feat.t())               # (B, n_items)
        score2 = None
        for i in range(self.n_behaviors):
            s = torch.mm(self._eval_proj[i][users], self._eval_item_prop[i].t())
            score2 = s if score2 is None else score2 + s
        score2 = score2 / self.n_behaviors
        return score1 + self.lamb * score2

    def clear_eval_cache(self):
        for a in ("_eval_item_prop", "_eval_proj", "_eval_user_feat", "_eval_item_feat"):
            if hasattr(self, a):
                delattr(self, a)


print("MBGCN (chuẩn) đã định nghĩa xong!")

MBGCN (chuẩn) đã định nghĩa xong!


In [7]:
# ============================================================
# CELL 7: DATASET BPR — triple (user, pos, neg) trên TARGET (purchase)
# ============================================================
class BPRDataset(Dataset):
    def __init__(self, R_target, num_items):
        coo = R_target.tocoo()
        self.users = coo.row.astype(np.int64)
        self.pos   = coo.col.astype(np.int64)
        self.num_items = num_items
        self.user_pos = defaultdict(set)
        for u, i in zip(self.users, self.pos):
            self.user_pos[u].add(i)
        print(f"BPRDataset: {len(self.users):,} positive (purchase) pairs")

    def __len__(self):
        return len(self.users)

    def __getitem__(self, idx):
        u = int(self.users[idx]); p = int(self.pos[idx])
        n = random.randint(0, self.num_items - 1)
        pos_set = self.user_pos[u]
        while n in pos_set:
            n = random.randint(0, self.num_items - 1)
        return u, p, n

train_dataset = BPRDataset(R_dict[TARGET], NUM_ITEMS)
train_loader = DataLoader(train_dataset, batch_size=2048, shuffle=True,
                          num_workers=2, pin_memory=True, drop_last=True)

BPRDataset: 1,308,730 positive (purchase) pairs


In [8]:
# ============================================================
# CELL 8: LOAD GROUND TRUTH + EXCLUDE MASK  (đúng protocol của dự án)
#   - {split}_ground_truth.pkl     : {user_idx -> [item_idx, ...]}
#   - train_mask_purchase_only.pkl : {user_idx -> [train purchase item_idx]}
# ============================================================
def load_pickle(path):
    with open(path, "rb") as f:
        return pickle.load(f)

val_gt  = {int(u): list(v) for u, v in load_pickle(f"{RAW_DIR}/val_ground_truth.pkl").items()}
test_gt = {int(u): list(v) for u, v in load_pickle(f"{RAW_DIR}/test_ground_truth.pkl").items()}
exclude_items = {int(u): list(v) for u, v in load_pickle(f"{RAW_DIR}/train_mask_purchase_only.pkl").items()}

val_users  = list(val_gt.keys())
test_users = list(test_gt.keys())
print(f"Val : {len(val_users):,} users | Test: {len(test_users):,} users")
print(f"Exclude mask (purchase-only): {len(exclude_items):,} users")

Val : 47,996 users | Test: 23,536 users
Exclude mask (purchase-only): 58,579 users


In [9]:
# ============================================================
# CELL 9: ĐÁNH GIÁ HR@k / NDCG@k  — khớp TemporalSplitEvaluator của dự án
#   (MBGCN không phải dot-product đơn thuần nên ta tính full-score qua model,
#    rồi áp ĐÚNG công thức HR/NDCG multi-positive + loại train items.)
# ============================================================
@torch.no_grad()
def evaluate_mbgcn(model, eval_user_ids, ground_truth, exclude_items,
                   n_items, ks, device, user_batch=512):
    model.prepare_eval()
    max_k = max(ks)
    ndcg_w = 1.0 / torch.log2(torch.arange(1, max_k + 1, device=device).float() + 1.0)
    sums = {f"{m}@{k}": 0.0 for m in ("HR", "NDCG") for k in ks}
    n_eval = len(eval_user_ids)

    for s in range(0, n_eval, user_batch):
        batch_uids = eval_user_ids[s:s + user_batch]
        B = len(batch_uids)
        u_t = torch.as_tensor(batch_uids, dtype=torch.long, device=device)
        scores = model.score_users(u_t)                       # (B, n_items)

        # loại bỏ train (purchase) items
        for r, u in enumerate(batch_uids):
            ex = exclude_items.get(u)
            if ex:
                scores[r, ex] = float("-inf")

        # ground-truth padded
        gt_lists = [ground_truth[u] for u in batch_uids]
        max_pos = max(len(x) for x in gt_lists)
        gt_pad = torch.full((B, max_pos), -1, dtype=torch.long, device=device)
        gt_cnt = torch.empty(B, dtype=torch.long, device=device)
        for r, items in enumerate(gt_lists):
            gt_cnt[r] = len(items)
            gt_pad[r, :len(items)] = torch.as_tensor(items, dtype=torch.long, device=device)

        top_idx = scores.topk(max_k, dim=-1).indices             # (B, max_k)
        hits = (top_idx.unsqueeze(-1) == gt_pad.unsqueeze(1)).any(dim=-1)   # (B, max_k)

        for k in ks:
            sums[f"HR@{k}"] += hits[:, :k].any(dim=-1).float().sum().item()
            dcg = (hits[:, :k].float() * ndcg_w[:k]).sum(dim=-1)
            ideal_len = torch.minimum(gt_cnt, torch.full_like(gt_cnt, k))
            idcg = torch.zeros_like(dcg)
            for ik in ideal_len.unique():
                m = ideal_len == ik
                if int(ik) > 0:
                    idcg[m] = ndcg_w[:int(ik)].sum()
            sums[f"NDCG@{k}"] += (dcg / idcg.clamp_min(1e-12)).sum().item()

    model.clear_eval_cache()
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return {k: v / n_eval for k, v in sums.items()}

print("evaluate_mbgcn sẵn sàng — metrics: HR@k, NDCG@k (full-rank, multi-positive).")

evaluate_mbgcn sẵn sàng — metrics: HR@k, NDCG@k (full-rank, multi-positive).


In [10]:
# ============================================================
# CELL 10: CONFIG + KHỞI TẠO MODEL  (config tham khảo MBGCN-Tmall)
# ============================================================
EMBED_SIZE       = 64
LAMB             = 1.0       # trọng số item-based CF (score2)
L2_NORM          = 1e-3
LEARNING_RATE    = 1e-3
NODE_DROPOUT     = 0.2
MESSAGE_DROPOUT  = 0.2
MGNN_WEIGHT      = [1.0, 1.0, 1.0]    # init weight cho [purchase, cart, view]
EPOCHS           = 40
EARLY_STOP_PATIENCE = 5
TOPK             = [1, 5, 10, 20, 50]
CHECKPOINT_DIR   = "./checkpoints_mbgcn"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

model = MBGCN(
    num_users=NUM_USERS, num_items=NUM_ITEMS, behaviors=BEHAVIORS,
    relation_dict=relation_dict, item_graph=item_graph,
    item_graph_degree=item_graph_degree,
    user_behaviour_degree=user_behaviour_degree, train_matrix=train_matrix,
    embed_size=EMBED_SIZE, lamb=LAMB, L2_norm=L2_NORM, mgnn_weight=MGNN_WEIGHT,
    node_dropout=NODE_DROPOUT, message_dropout=MESSAGE_DROPOUT, device=device,
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

n_param = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model: MBGCN (chuẩn) | embed={EMBED_SIZE} (feature dim hiệu dụng = {EMBED_SIZE*2}) | λ={LAMB}")
print(f"Users={NUM_USERS:,} | Items={NUM_ITEMS:,} | behaviors={BEHAVIORS}")
print(f"Trainable params: {n_param:,}")

Model: MBGCN (chuẩn) | embed=64 (feature dim hiệu dụng = 128) | λ=1.0
Users=203,063 | Items=29,892 | behaviors=['purchase', 'cart', 'view']
Trainable params: 14,974,659


In [11]:
# ============================================================
# CELL 11: TRAINING LOOP  (BPR mean + L2/batch_size — đúng loss.py gốc)
# ============================================================
best_ndcg, best_epoch, patience = 0.0, 0, 0
history = []
BATCH = train_loader.batch_size

print("=" * 64); print("BẮT ĐẦU HUẤN LUYỆN MBGCN"); print("=" * 64)

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    t0 = time.time()
    for u, p, n in train_loader:
        user = u.to(device).unsqueeze(-1)                    # (B,1)
        item = torch.stack([p, n], dim=1).to(device)         # (B,2) -> [pos, neg]
        optimizer.zero_grad()
        scores, L2 = model(user, item)                       # scores (B,2)
        bpr = -torch.log(torch.sigmoid(scores[:, 0] - scores[:, 1]) + 1e-8).mean()
        loss = bpr + L2 / BATCH
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    val = evaluate_mbgcn(model, val_users, val_gt, exclude_items,
                         NUM_ITEMS, TOPK, device)
    ndcg20, hr20 = val['NDCG@20'], val['HR@20']
    mw = F.softmax(model.mgnn_weight, dim=0).detach().cpu().numpy()
    mw_str = ', '.join(f"{b}={w:.3f}" for b, w in zip(BEHAVIORS, mw))
    print(f"Epoch {epoch:02d}/{EPOCHS} | Loss {avg_loss:.4f} | "
          f"HR@20 {hr20:.4f} | NDCG@20 {ndcg20:.4f} | "
          f"mgnn[{mw_str}] | {time.time()-t0:.1f}s")
    history.append({'epoch': epoch, 'loss': avg_loss, **val})

    if ndcg20 > best_ndcg:
        best_ndcg, best_epoch, patience = ndcg20, epoch, 0
        torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(),
                    'val_results': val}, f"{CHECKPOINT_DIR}/mbgcn_best.pth")
        print(f"  => Saved best (NDCG@20={ndcg20:.4f})")
    else:
        patience += 1
        if patience >= EARLY_STOP_PATIENCE:
            print(f"\nEarly stopping @ epoch {epoch}. Best epoch {best_epoch}.")
            break

print(f"\nXong! Best NDCG@20={best_ndcg:.4f} @ epoch {best_epoch}")

BẮT ĐẦU HUẤN LUYỆN MBGCN
Epoch 01/40 | Loss 3.6708 | HR@20 0.2390 | NDCG@20 0.0758 | mgnn[purchase=0.303, cart=0.269, view=0.428] | 77.7s
  => Saved best (NDCG@20=0.0758)
Epoch 02/40 | Loss 0.2109 | HR@20 0.2482 | NDCG@20 0.0780 | mgnn[purchase=0.303, cart=0.266, view=0.431] | 76.3s
  => Saved best (NDCG@20=0.0780)
Epoch 03/40 | Loss 0.0863 | HR@20 0.2218 | NDCG@20 0.0685 | mgnn[purchase=0.302, cart=0.264, view=0.434] | 76.3s
Epoch 04/40 | Loss 0.0557 | HR@20 0.2130 | NDCG@20 0.0654 | mgnn[purchase=0.301, cart=0.261, view=0.438] | 76.3s
Epoch 05/40 | Loss 0.0475 | HR@20 0.2152 | NDCG@20 0.0660 | mgnn[purchase=0.300, cart=0.261, view=0.439] | 76.4s
Epoch 06/40 | Loss 0.0445 | HR@20 0.2076 | NDCG@20 0.0611 | mgnn[purchase=0.300, cart=0.260, view=0.440] | 76.3s
Epoch 07/40 | Loss 0.0437 | HR@20 0.2045 | NDCG@20 0.0596 | mgnn[purchase=0.300, cart=0.260, view=0.440] | 76.4s

Early stopping @ epoch 7. Best epoch 2.

Xong! Best NDCG@20=0.0780 @ epoch 2


In [12]:
# ============================================================
# CELL 12: ĐÁNH GIÁ TRÊN TEST SET
# ============================================================
ckpt = torch.load(f"{CHECKPOINT_DIR}/mbgcn_best.pth", map_location=device, weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
print(f"Loaded best model @ epoch {ckpt['epoch']}")

test_results = evaluate_mbgcn(model, test_users, test_gt, exclude_items,
                              NUM_ITEMS, TOPK, device)
print("\n" + "=" * 50); print("KẾT QUẢ TEST SET"); print("=" * 50)
for m, v in test_results.items():
    print(f"  {m:<10}: {v:.4f}")

mw = F.softmax(model.mgnn_weight, dim=0).detach().cpu().numpy()
print("\nBehavior importance (softmax mgnn_weight):")
for b, w in zip(BEHAVIORS, mw):
    print(f"  {b:8s}: {w:.4f}")

Loaded best model @ epoch 2

KẾT QUẢ TEST SET
  HR@1      : 0.0213
  HR@5      : 0.0721
  HR@10     : 0.1119
  HR@20     : 0.1648
  HR@50     : 0.2505
  NDCG@1    : 0.0213
  NDCG@5    : 0.0335
  NDCG@10   : 0.0434
  NDCG@20   : 0.0551
  NDCG@50   : 0.0711

Behavior importance (softmax mgnn_weight):
  purchase: 0.3033
  cart    : 0.2659
  view    : 0.4308


In [13]:
# ============================================================
# CELL 14: LƯU KẾT QUẢ
# ============================================================
results_summary = {
    'model': 'MBGCN (tsinghua-fib-lab, chuẩn)',
    'dataset': 'rees46-bpatmp-temporal',
    'behaviors': BEHAVIORS, 'target': TARGET,
    'embed_size': EMBED_SIZE, 'feature_dim': EMBED_SIZE * 2,
    'lambda': LAMB, 'item_graph_topk': ITEM_GRAPH_TOPK,
    'best_epoch': best_epoch, 'best_val_ndcg20': best_ndcg,
    'test_results': test_results,
    'behavior_importance': {
        b: float(w) for b, w in
        zip(BEHAVIORS, F.softmax(model.mgnn_weight, dim=0).detach().cpu().numpy())
    },
}
with open(f"{CHECKPOINT_DIR}/mbgcn_results.json", "w") as f:
    json.dump(results_summary, f, indent=2, ensure_ascii=False)
print("Đã lưu:", f"{CHECKPOINT_DIR}/mbgcn_results.json")
print(json.dumps(results_summary, indent=2, ensure_ascii=False))

Đã lưu: ./checkpoints_mbgcn/mbgcn_results.json
{
  "model": "MBGCN (tsinghua-fib-lab, chuẩn)",
  "dataset": "rees46-bpatmp-temporal",
  "behaviors": [
    "purchase",
    "cart",
    "view"
  ],
  "target": "purchase",
  "embed_size": 64,
  "feature_dim": 128,
  "lambda": 1.0,
  "item_graph_topk": 10,
  "best_epoch": 2,
  "best_val_ndcg20": 0.07795951171501128,
  "test_results": {
    "HR@1": 0.021329027872195785,
    "HR@5": 0.07205982324949015,
    "HR@10": 0.1118711760707002,
    "HR@20": 0.164768864717879,
    "HR@50": 0.25046736913664175,
    "NDCG@1": 0.021329027872195785,
    "NDCG@5": 0.033539609387162425,
    "NDCG@10": 0.04342443208804672,
    "NDCG@20": 0.05512258732586309,
    "NDCG@50": 0.07112970123641107
  },
  "behavior_importance": {
    "purchase": 0.30333006381988525,
    "cart": 0.2658684551715851,
    "view": 0.43080148100852966
  }
}
